In [ ]:
from typo_utils import make_typo, ngram_overlap, evaluate_ngram_overlap_mecab, compute_micro_average, ipadic_files
from typo_utils import normalize_unicode_symbol_white, benchmark_one, show_debug, compute_metrics
import MeCab
import pandas as pd
from tqdm.notebook import tqdm


In [ ]:
import re
import requests

def strip_gpt_oss_channels(s: str) -> str:
    """llama-server(gpt-oss)がcontentに混ぜる <|channel|>... を除去して本文だけ返す"""
    if not s:
        return ""

    # final チャンネルがある場合は final だけ採用
    m = re.search(r"<\|channel\|>final<\|message\|>(.*?)(?:<\|end\|>|$)", s, re.DOTALL)
    if m:
        return m.group(1).strip()

    # analysis + <|end|> + 本文 の形式なら <|end|> 以降だけ採用
    if "<|end|>" in s:
        return s.split("<|end|>")[-1].strip()

    # それ以外：タグを雑に除去
    s = re.sub(r"<\|[^>]*\|>", "", s)
    return s.strip()

def extract_last_report(text: str) -> str:
    """テンプレが2回出る等の事故に備え、最後のヘッダ以降だけ採用"""
    if not text:
        return ""
    header = "【乳癌取扱い規約・第19版.】"
    idxs = [m.start() for m in re.finditer(re.escape(header), text)]
    if not idxs:
        return text.strip()
    return text[idxs[-1]:].strip()

def fix_with_llama_server_chat(
    typo_text: str,
    server_url: str = "http://127.0.0.1:8080/v1/chat/completions",
) -> str:
    system_prompt = """あなたは病理診断文の校正者です。
    【厳守】
    - 校正された文章のみを1回出力する
    - 前置き、解説、注釈、英語、Markdown、箇条書き、コードブロック、思考過程を一切出力しない
    - 誤字、脱字、文字抜け、不要な文字の追加、文字の交換、漢字変換ミスのみ修正する
    - 言い回しや表現の修正、句読点の修正、改行位置の変更はしない
    """
    
    user_prompt = f"""次の文章は病理診断の病理所見です。
    誤字、脱字、文字抜け、不要な文字の追加、文字の交換、漢字変換ミスを修正し、出力してください。
    
    元の文章：
    {typo_text}
    校正後の文章："""

    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }

    try:
        r = requests.post(
            server_url,
            json=payload,
            proxies={"http": None, "https": None},
            timeout=1000,
        )
        r.raise_for_status()
        data = r.json()

        content = (data.get("choices", [{}])[0].get("message", {}) or {}).get("content", "")
        content = content.strip()

        # gpt-oss特有のチャンネルタグ等を除去
        content = strip_gpt_oss_channels(content)

        return content

    except requests.exceptions.RequestException as e:
        print("❌ LLM接続エラー:", e)
        # 可能ならレスポンス本文も出す
        try:
            print("response text:", r.text)  # type: ignore
        except Exception:
            pass
        return ""

In [ ]:
dic_dir = "./dictionaries/mecab-ipadic-2.7.0-20070610"
usr_dir = "./dictionaries/user.dic"

tagger = MeCab.Tagger(f'-r /dev/null -d "{dic_dir}" -u "{usr_dir}"')

df_path = "./reports.tsv"
df = pd.read_csv(df_path, sep="\t")

In [ ]:
import random
count = 0
max_count = 50
index = list(range(len(df["所見"]) ))
random.shuffle(index)
typo_rate = 0.1
if 'original' not in df.columns:
    df['original'] = ""
if 'typo_text' not in df.columns:
    df['typo_text'] = ""
if 'typo_log' not in df.columns:
    df['typo_log'] = ""
for progress, i in tqdm(enumerate(index, start=1), total=len(index)):
    if count >= max_count:
        break
    original = f'{df["所見"][i]}。'
    if len(original) >= 100 and len(original) < 400 and (('P24' in df["標本番号_x"][i]) or ('O24' in df["標本番号_x"][i])):
        original = normalize_unicode_symbol_white(original)
        original = make_typo(original, 0.0, dic_dir, usr_dir)[1]
        _, typo, typo_log = make_typo(original, typo_rate, dic_dir, usr_dir)
    
        df.at[i, 'original'] = original
        df.at[i, 'typo_text'] = typo
        df.at[i, 'typo_log'] = typo_log
    
        count += 1
    else:
        continue
    
      
# original が空でない行だけ抽出
df_valid = df[df['original'] != ""]

# CSVとして保存
df_valid.to_csv(f"typo_{max_count}cases.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_path = "./typo_50cases.csv"
df = pd.read_csv(df_path)

In [ ]:
import time
results_mecab_with_counts = []
results_char_with_counts = []
index = list(range(len(df["所見"]) ))
llama_times = []

from llama_utils import get_llama_server_models
model_name = get_llama_server_models("http://localhost:8080")["data"][0]["id"]

if f'{model_name}' not in df.columns:
    df[f'{model_name}'] = ""

for progress, i in tqdm(enumerate(index, start=1), total=len(index) ):
  for text_type in ["所見"]:
    if text_type == "臨床所見":
        original = f'臨床所見:{df["患者年齢(年)_x"][i]}歳、{df["性別名_x"][i]}。{df["臨床所見"][i]}。'
    else:
        original = f'{df["original"][i]}'
        typo = f'{df["typo_text"][i]}'
        typo_log = f'{df["typo_log"][i]}'
            
    t0 = time.perf_counter()
    fixed = fix_with_llama_server_chat(typo, server_url="http://localhost:8080/v1/chat/completions")
    t1 = time.perf_counter()
    llama_times.append(t1 - t0)

    fixed = normalize_unicode_symbol_white( fixed )

    df.at[i, f'{model_name}'] = fixed

    # MeCabベースのn-gram評価
    metrics_mecab = evaluate_ngram_overlap_mecab(
        ref_text=original,
        pred_text=fixed,
        tagger=tagger,
        n=2,
        return_counts=True
    )
    results_mecab_with_counts.append(metrics_mecab)

    # 文字n-gram評価
    metrics_char = ngram_overlap(
        ref=original,
        pred=fixed,
        n=3,
        return_counts=True
    )
    results_char_with_counts.append(metrics_char)

    # ログ表示
    if progress==3:
        compute_metrics(results_mecab_with_counts, results_char_with_counts)
        print(f"[{progress}] オリジナル:")
        print(original)
        print(f"[{progress}] タイポ:")
        print(typo)
        print(f"[{progress}] タイポ修正例:")
        print(fixed)
        print()
        
compute_metrics(results_mecab_with_counts, results_char_with_counts)
try:
    df.to_csv(f"{model_name.split('/')[1]}_typo_30cases.csv", index=False, encoding="utf-8-sig")
except:
    df.to_csv(f"{model_name}_typo_30cases.csv", index=False, encoding="utf-8-sig")    

In [ ]:
import numpy as np
print(f"llama平均応答時間: {np.mean(llama_times[:30]):.3f} 秒")
print(f"llama中央値応答時間: {np.median(llama_times[:30]):.3f} 秒")
print(f"llama最小応答時間: {np.min(llama_times[:30]):.3f} 秒")
print(f"llama最大応答時間: {np.max(llama_times[:30]):.3f} 秒")